# CIFAR10 e MNIST CNN: Ablação
Nesta versão do notebook, realizamos uma ablação do modelo CNN, removendo camadas e comparando o desempenho. O objetivo é entender a importância de cada componente da arquitetura para a tarefa de classificação.

Escolhido (no momento): BatchNorm. A ideia é treinar o modelo sem BatchNorm e comparar os resultados com o modelo   .     

In [ ]:
from pathlib import Path
import json

import torch
import torchvision
import torch.nn as nn
import torchvision.transforms as transforms

from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support

from sklearn.model_selection import KFold
from torch.utils.data import DataLoader

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo em uso: {device}")

Dispositivo em uso: cuda


In [ ]:
# Ajuste estes caminhos conforme seu ambiente
DATA_ROOT = Path("data")
OUTPUT_JSON = Path("../results") / "ablacao_batchnorm.json"
NUM_WORKERS = 2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo em uso: {device}")

OUTPUT_JSON.parent.mkdir(parents=True, exist_ok=True)

Dispositivo em uso: cuda


### Classe do Modelo

In [ ]:
class CNN_Ablacao(nn.Module):  # Versão com controle de BatchNorm
    def __init__(self, input_shape, block_configs, fc_sizes, dropout_rate, activation_func,
                 conv_kernel_size=3, conv_stride=1, conv_padding=1, pool_kernel_size=2, use_batch_norm=True):
        super(CNN_Ablacao, self).__init__()

        layers = []
        in_channels = input_shape[0]

        for num_convs, out_channels in block_configs:
            for i in range(num_convs):
                layers.append(nn.Conv2d(
                    in_channels,
                    out_channels,
                    kernel_size=conv_kernel_size,
                    stride=conv_stride,
                    padding=conv_padding
                ))

                if i == 0 and use_batch_norm:  # BatchNorm apenas na primeira conv de cada bloco
                    layers.append(nn.BatchNorm2d(out_channels))

                layers.append(activation_func)
                in_channels = out_channels

            layers.append(nn.MaxPool2d(kernel_size=pool_kernel_size))
            layers.append(nn.Dropout2d(dropout_rate))

        self.features = nn.Sequential(*layers)

        with torch.no_grad():
            dummy_input = torch.zeros(1, *input_shape)
            try:
                dummy_output = self.features(dummy_input)
                flattened_size = dummy_output.numel()
            except RuntimeError as e:
                raise ValueError(f"As configurações encolheram a dimensão espacial a zero ou menos. Erro: {e}")

        fc_layers = []
        in_f = flattened_size
        for h_size in fc_sizes:
            fc_layers.append(nn.Linear(in_f, h_size))
            fc_layers.append(activation_func)
            fc_layers.append(nn.Dropout(dropout_rate))
            in_f = h_size

        fc_layers.append(nn.Linear(in_f, 10))
        self.classifier = nn.Sequential(*fc_layers)

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

### Carregamento de Dados

In [5]:
cifar_transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

mnist_transform = transforms.Compose(
    [transforms.Grayscale(num_output_channels=3),
     transforms.Resize((32, 32)),
     transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

# Dataset CIFAR10

c10_train_dataset = torchvision.datasets.CIFAR10(root=str(DATA_ROOT), train=True,
                                        download=True, transform=cifar_transform)
c10_test_dataset = torchvision.datasets.CIFAR10(root=str(DATA_ROOT), train=False,
                                       download=True, transform=cifar_transform)
c10_class_names = list(c10_train_dataset.classes)


# Dataset MNIST

mnist_train_dataset = torchvision.datasets.MNIST(root=str(DATA_ROOT), train=True,
                                        download=True, transform=mnist_transform)
mnist_test_dataset = torchvision.datasets.MNIST(root=str(DATA_ROOT), train=False,
                                       download=True, transform=mnist_transform)
mnist_class_names = [str(i) for i in range(10)]

d:\programming\26.1\IAP\.venv\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


### Funções Auxiliares

In [ ]:
import random
from tqdm import tqdm
import numpy as np

def set_seeds(seed):
    """Define todas as seeds para reprodutibilidade"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [6]:
def _collect_predictions(model, dataloader):
    model.eval()
    all_preds = []
    all_targets = []
    with torch.no_grad():
        for images, targets in tqdm(dataloader, desc="Inferencia"):
            images = images.to(device)
            logits = model(images)
            preds = torch.argmax(logits, dim=1).cpu().tolist()
            all_preds.extend(preds)
            all_targets.extend(targets.tolist())
    return {"preds": all_preds, "targets": all_targets}

def evaluate_dataset(model, dataloader, class_names, dataset_name):
    results = _collect_predictions(model, dataloader)
    preds = results["preds"]
    targets = results["targets"]
    acc = accuracy_score(targets, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        targets, preds, average="weighted", zero_division=0
    )
    conf_matrix = confusion_matrix(targets, preds).tolist()
    return {
        "dataset": dataset_name,
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "confusion_matrix": conf_matrix,
        "class_names": class_names,
    }

In [7]:
def train_one_epoch(model, dataloader, optimizer, criterion):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for images, targets in tqdm(dataloader, desc="Treino", leave=False):
        images = images.to(device)
        targets = targets.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        preds = torch.argmax(logits, dim=1)
        correct += (preds == targets).sum().item()
        total += targets.size(0)
    return running_loss / total, correct / total

In [8]:
def train_model(model: nn.Module, train_loader: DataLoader, optimizer, criterion, epochs: int, label: str):
    for epoch in range(epochs):
        loss, acc = train_one_epoch(model, train_loader, optimizer, criterion)
        if epoch == 0 or (epoch + 1) % 10 == 0 or (epoch + 1) == epochs:
            print(f"{label} - epoch {epoch + 1}/{epochs} - loss: {loss:.4f} - acc: {acc:.4f}")

In [ ]:
def build_cnn_ablacao_from_params(params: dict, input_shape: tuple, use_batch_norm: bool = True):
    """Constrói CNN com opção de ativar/desativar BatchNorm"""
    block_configs = params["arch_style"]
    fc_sizes = [params[f"fc_size_l{i}"] for i in range(params["n_fc_layers"])]
    activation = params["activation"]
    return CNN_Ablacao(
        input_shape=input_shape,
        block_configs=block_configs,
        fc_sizes=fc_sizes,
        dropout_rate=params["dropout"],
        activation_func=activation,
        conv_kernel_size=params["conv_kernel_size"],
        use_batch_norm=use_batch_norm,
    )


def build_adam_optimizer(model: nn.Module, lr: float, weight_decay: float):
    return torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

### Avaliação

In [ ]:
cnn_params = {
    "input_shape": (3, 32, 32),
    "arch_style": [(2, 32), (2, 64), (2, 128)],
    "n_fc_layers": 1,
    "fc_size_l0": 2048,
    "conv_kernel_size": 3,
    "lr": 1e-4,
    "weight_decay": 1e-4,
    "dropout": 0.2,
    "batch_size": 32,
    "epochs": 50,
    "optimizer": "adam",
    "criterion": nn.CrossEntropyLoss(),
    "activation": nn.ReLU(),
}

Parametros CNN: {'input_shape': (3, 32, 32), 'arch_style': [(2, 32), (2, 64), (2, 128)], 'n_fc_layers': 1, 'fc_size_l0': 2048, 'conv_kernel_size': 3, 'lr': 0.0001, 'weight_decay': 0.0001, 'dropout': 0.2, 'batch_size': 32, 'epochs': 50, 'optimizer': 'adam', 'criterion': CrossEntropyLoss(), 'activation': ReLU()}


In [ ]:


epochs = cnn_params["epochs"]
criterion = cnn_params["criterion"]
input_shape = cnn_params["input_shape"]
batch_size = cnn_params["batch_size"]

c10_train_loader = DataLoader(
    c10_train_dataset, batch_size=batch_size, shuffle=True, num_workers=NUM_WORKERS
 )
c10_test_loader = DataLoader(
    c10_test_dataset, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS
 )
mnist_train_loader = DataLoader(
    mnist_train_dataset, batch_size=batch_size, shuffle=True, num_workers=NUM_WORKERS
 )
mnist_test_loader = DataLoader(
    mnist_test_dataset, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS
 )

Treinando CNN no CIFAR-10...


CIFAR-10 - epoch 1/50 - loss: 1.5095 - acc: 0.4504


CIFAR-10 - epoch 10/50 - loss: 0.4870 - acc: 0.8294


CIFAR-10 - epoch 20/50 - loss: 0.1988 - acc: 0.9309


CIFAR-10 - epoch 30/50 - loss: 0.1004 - acc: 0.9665


...

MNIST - epoch 10/50 - loss: 0.0151 - acc: 0.9951


MNIST - epoch 20/50 - loss: 0.0079 - acc: 0.9973


MNIST - epoch 30/50 - loss: 0.0063 - acc: 0.9979


MNIST - epoch 40/50 - loss: 0.0052 - acc: 0.9984


MNIST - epoch 50/50 - loss: 0.0046 - acc: 0.9986


### Treino para Ablação
Treinar múltiplas vezes com diferentes seeds aleatórios para comparar o impacto de remover BatchNorm

In [ ]:


# Configuração da ablação
seeds = [42, 123, 456]  # 3 execuções com seeds diferentes
final_results = {
    "com_batchnorm": [],
    "sem_batchnorm": []
}
datasets = {
    "CIFAR-10": {"train": c10_train_loader, "test": c10_test_loader},
    "MNIST": {"train": mnist_train_loader, "test": mnist_test_loader}
}


for i, seed in enumerate(seeds):
    for dataset in datasets:
        print(f"\n--- Execução {i+1}/{len(seeds)} (seed={seed}) ---\n")
        
        # Treino COM BatchNorm
        set_seeds(seed)
        model_bn = build_cnn_ablacao_from_params(cnn_params, input_shape, use_batch_norm=True).to(device)
        optimizer_bn = build_adam_optimizer(model_bn, cnn_params["lr"], cnn_params["weight_decay"])
        
        print(f"Treinando {dataset} COM BatchNorm...")
        train_model(model_bn, datasets[dataset]["train"], optimizer_bn, criterion, epochs, f"[BN] {dataset} (seed {seed})")
        metrics_bn = evaluate_dataset(model_bn, datasets[dataset]["test"], c10_class_names, dataset)
        final_results["com_batchnorm"].append(metrics_bn)
        
        # Treino SEM BatchNorm
        set_seeds(seed)
        model_no_bn = build_cnn_ablacao_from_params(cnn_params, input_shape, use_batch_norm=False).to(device)
        optimizer_no_bn = build_adam_optimizer(model_no_bn, cnn_params["lr"], cnn_params["weight_decay"])
        
        print(f"\nTreinando {dataset} SEM BatchNorm...")
        train_model(model_no_bn, datasets[dataset]["train"], optimizer_no_bn, criterion, epochs, f"[NoBN] {dataset} (seed {seed})")
        metrics_no_bn = evaluate_dataset(model_no_bn, datasets[dataset]["test"], c10_class_names, dataset)
        final_results["sem_batchnorm"].append(metrics_no_bn)




### Exportar Resultados

In [ ]:

with open(OUTPUT_JSON, "w") as f:
    json.dump(final_results, f, indent=4)
print(f"Resultados salvos em: {OUTPUT_JSON}")

Inferencia: 100%|██████████| 313/313 [00:06<00:00, 48.06it/s] 

Resultados salvos em: ..\results\test_metrics.json
